# Train PointNu-Net on Kaggle PanNuke

Notebook này hướng dẫn chạy train PointNu-Net trên Kaggle theo đúng kiểu notebook: clone source trong notebook, cài dependencies, rồi train trực tiếp từ PanNuke trong `/kaggle/input`.

Notebook được viết cho đúng cấu trúc bạn chụp: `/kaggle/input/PanNuke/fold_1/Fold 1/images/fold1/images.npy` và `/kaggle/input/PanNuke/fold_1/Fold 1/masks/fold1/masks.npy`. Không cần copy hay symlink dữ liệu sang repo clone nữa.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

KAGGLE_INPUT = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/Kaiseem/PointNu-Net.git'
REPO_DIR = WORK_DIR / 'PointNu-Net'
DATASET_ROOT = KAGGLE_INPUT / 'PanNuke'

print('Kaggle input exists:', KAGGLE_INPUT.exists())
print('Dataset root:', DATASET_ROOT)
print('Dataset root exists:', DATASET_ROOT.exists())
print('Working dir:', WORK_DIR)
print('Repo dir:', REPO_DIR)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print('Current directory:', os.getcwd())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Dependencies installed')

In [ ]:
source_root = DATASET_ROOT
print('Using PanNuke root directly:', source_root)
print('Checking expected Kaggle structure:')

# Exact structure based on your screenshot:
# /kaggle/input/PanNuke/fold_1/Fold 1/images/fold1/images.npy
# /kaggle/input/PanNuke/fold_1/Fold 1/masks/fold1/masks.npy
for fold in [1, 2, 3]:
    expected_img = source_root / f'fold_{fold}' / f'Fold {fold}' / 'images' / f'fold{fold}' / 'images.npy'
    expected_mask = source_root / f'fold_{fold}' / f'Fold {fold}' / 'masks' / f'fold{fold}' / 'masks.npy'
    print(f'Fold {fold}:')
    print('  expected image:', expected_img)
    print('  expected mask :', expected_mask)
    print('  image exists  :', expected_img.exists())
    print('  mask exists   :', expected_mask.exists())

print('No copy/symlink step is needed now.')

In [ ]:
# Chay train voi cau hinh mac dinh cho PanNuke
# Neu bi OOM, hay mo configs/pannuke.yaml va giam train.batch_size truoc khi chay cell nay.
subprocess.run([
    sys.executable,
    'train_pannuke.py',
    '--name=kaggle_pannuke',
    '--seed=888',
    '--train_fold=1',
    '--val_fold=2',
    '--test_fold=3'
], check=True)

## Sau khi train

Checkpoint và log sẽ được lưu trong `outputs/kaggle_pannuke/`. Nếu bạn muốn thử fold khác, chỉ cần đổi ba tham số `--train_fold`, `--val_fold`, `--test_fold` theo một hoán vị của 1, 2, 3.

Vì `configs/pannuke.yaml` đã trỏ trực tiếp tới `/kaggle/input/PanNuke`, notebook này không còn cần bước mapping/copy dữ liệu nữa.